# Froth Flotation Survival Analysis

This notebook applies a **Cox Proportional Hazards model** with **bootstrap LASSO** regularization to mineral particle flotation data.

Flotation is framed as a survival problem:
- **Event (E=1/Death)**: particle is recovered into concentrate (CA, CB, CC, CD streams)
- **Censored (E=0/Survived)**: particle reports to tailings (TD) — not recovered
- **Time (T)**: cumulative flotation time at which recovery occurs

The model identifies which particle properties (mineralogy, surface composition, size) drive faster or slower flotation.

## 1. Setup & Imports

In [ ]:
# Install dependencies (uncomment if running on Colab)
# !pip install lifelines

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_data, balance_classes, map_survival_variables
from src.feature_engineering import select_features, remove_collinear
from src.model import fit_bootstrap_cox, fit_final_model, save_bootstrap_results
from src.visualization import (
    plot_feature_impact,
    plot_baseline_hazard,
    plot_floatability_boxplot,
    plot_r2_histogram
)

print('All modules loaded successfully.')

## 2. Load & Prepare Data

Load the particle dataset, balance classes to ~60k per stream, and map class labels to survival variables (Time and Event).

In [ ]:
# Load raw data
df = load_data('Traindata_20um.csv')

In [ ]:
# Balance classes to 300k total (~60k per stream)
df = balance_classes(df, target_total_n=300000)

In [ ]:
# Map class labels to survival variables
# CA -> T=0.75, E=1 | CB -> T=1.5, E=1 | CC -> T=3.0, E=1 | CD -> T=6.0, E=1 | TD -> T=6.0, E=0
df = map_survival_variables(df)

## 3. Feature Engineering

Select numeric features, drop any constant columns, then iteratively remove multicollinear features using VIF (threshold > 10).

In [ ]:
# Select numeric features and drop constants
feature_cols, data_model = select_features(df)

In [ ]:
# Remove collinear features (VIF > 10)
print('Checking collinearity...')
model_data = remove_collinear(data_model)
final_features = [c for c in model_data.columns if c not in ['T', 'E']]
print(f'\nFeatures after VIF ({len(final_features)}): {final_features}')

## 4. Bootstrap Cox PH Model Fitting

Fit a LASSO-regularized Cox PH model using 1000 bootstrap iterations (50k subsamples each). This produces stable coefficient estimates with confidence intervals.

In [ ]:
# Run bootstrap fitting (this takes ~40 minutes with 2 CPUs)
boot_results = fit_bootstrap_cox(
    model_data,
    n_bootstrap=1000,
    subsample_n=50000,
    penalizer=0.02,
    l1_ratio=1.0
)

In [ ]:
# Save bootstrap results to CSV files
save_bootstrap_results(boot_results)

In [ ]:
# Fit final model on full dataset with bootstrap-averaged coefficients
cph_final = fit_final_model(model_data, boot_results)

## 5. Results & Visualization

Visualize the model results: feature impacts (forest plot), baseline cumulative hazard, and predicted floatability by stream and dominant mineral.

In [ ]:
# Forest plot: feature impact on flotation rate
plot_feature_impact(cph_final)

In [ ]:
# Baseline cumulative hazard (kinetic rate proxy)
plot_baseline_hazard(cph_final)

In [ ]:
# Faceted boxplot: predicted floatability by stream and dominant mineral
plot_floatability_boxplot(df, cph_final, model_data)

## 6. Model Validation (R² Goodness-of-Fit)

Compute R² for individual particle kinetic fits to validate that the survival model reproduces particle-level recovery behavior.

In [ ]:
import numpy as np

# Compute individual R² values (vectorized)
hazards = cph_final.predict_partial_hazard(model_data)
times = np.array([0.75, 1.5, 3.0, 6.0])
baseline_H = cph_final.baseline_cumulative_hazard_

# Interpolate baseline cumulative hazard at observation times
H0_at_times = np.interp(times, baseline_H.index, baseline_H.values.flatten())

# Predicted survival: S(t) = exp(-H0(t) * exp(beta*X))
y_pred = np.exp(-np.outer(hazards.values.flatten(), H0_at_times))

# Observed survival (step function based on event time)
T_vals = model_data['T'].values
E_vals = model_data['E'].values
y_obs = np.zeros((len(T_vals), len(times)))
for j, t in enumerate(times):
    y_obs[:, j] = np.where((T_vals <= t) & (E_vals == 1), 0.0, 1.0)

# R² per particle
y_mean = y_obs.mean(axis=1, keepdims=True)
ss_res = np.sum((y_obs - y_pred)**2, axis=1)
ss_tot = np.sum((y_obs - y_mean)**2, axis=1)
r2_values = 1 - (ss_res / ss_tot)
r2_values[ss_tot == 0] = 1.0

# Plot R² distribution
plot_r2_histogram(r2_values)

## 7. Load Saved Model (Optional)

If you have a previously saved model and bootstrap results, you can reload them without re-fitting.

In [ ]:
import pickle
import pandas as pd
from lifelines import CoxPHFitter

# Uncomment to load a previously saved model:
# with open('cox_model_lasso.pkl', 'rb') as f:
#     cph_final = pickle.load(f)
#
# bootstrap = pd.read_csv('bootstrap_summary.csv', index_col=0)
# mean_coefs = bootstrap['Mean Coef']
# cph_final.params_ = mean_coefs.reindex(cph_final.params_.index, fill_value=0.0)
#
# print('Model loaded with bootstrap coefficients!')
# cph_final.print_summary()